## 1. Notebook Structure & Parameters
Use this section to register shared parameters and helper stubs before executing the pipeline.

In [ ]:
from __future__ import annotations

# ---- Required configuration placeholders ----
PROJECT_ID = "YOUR_GCP_PROJECT_ID"
REGION = "us-central1"
BUCKET_NAME = "your-shared-artifacts-bucket"
LOCAL_DATA_ROOT = "./outputs"  # directory containing shards from Colab sessions
SHARD_PATTERN = "train_text_embeddings_part*.npy"

# ---- Utility stubs (fill in as needed) ----
def describe_environment():
    """Print diagnostic info for debugging runtimes."""
    import platform
    print(f"Python: {platform.python_version()}")
    print(f"Platform: {platform.platform()}")


def list_shards(pattern: str) -> list[str]:
    """Return sorted shard paths matching the pattern."""
    from glob import glob
    return sorted(glob(pattern))


describe_environment()
print("Discovered shards:", len(list_shards(f"{LOCAL_DATA_ROOT}/{SHARD_PATTERN}")))

## 2. Install & Import Dependencies
Run once per runtime to provision Google Cloud libraries and supporting tooling.

In [ ]:
%%capture
!pip install --quiet pandas numpy scikit-learn lightgbm sentence-transformers transformers torch torchvision Pillow

In [ ]:
%%capture
!pip install --quiet google-cloud-storage google-auth google-auth-oauthlib tqdm

In [ ]:
from google.cloud import storage
from google.oauth2 import service_account
from google.auth.transport.requests import Request
import google.auth

from pathlib import Path
import json
from tqdm import tqdm

print("Google Cloud client loaded")

## 3. Authenticate with the Google Cloud Free Tier
Choose one option below based on the credentials you control (service account JSON or `gcloud` CLI login).

In [ ]:
# Option A: Use an uploaded service-account key (recommended for automation)
SERVICE_ACCOUNT_JSON = "path/to/service-account.json"  # TODO: replace or leave None

if Path(SERVICE_ACCOUNT_JSON).is_file():
    credentials = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_JSON,
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
else:
    # Option B: Use the Colab metadata / gcloud auth login flow
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    if credentials.requires_scopes:
        credentials = credentials.with_scopes(["https://www.googleapis.com/auth/cloud-platform"])
    if credentials.expired and credentials.refresh_token:
        credentials.refresh(Request())

client = storage.Client(project=PROJECT_ID, credentials=credentials)
print("Authenticated client for project:", client.project)

## 4. Configure Cloud Storage Bucket
Create or reuse a bucket that respects free-tier allowances (multi-regional storage is not eligible).

In [ ]:
def ensure_bucket(client: storage.Client, bucket_name: str, region: str) -> storage.Bucket:
    bucket = client.bucket(bucket_name)
    if bucket.exists():
        LOGGER = logging.getLogger("bucket")
        LOGGER.info("Bucket %s already exists", bucket_name)
        bucket.reload()
        return bucket

    bucket.storage_class = "STANDARD"
    new_bucket = client.create_bucket(bucket, location=region)
    new_bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    new_bucket.patch()
    print(f"Created bucket {bucket_name} in {region}")
    return new_bucket


bucket = ensure_bucket(client, BUCKET_NAME, REGION)
print("Bucket storage class:", bucket.storage_class)

## 5. Bulk Upload Dataset Shards
This helper walks the local shard directory and pushes files in manageable batches with progress feedback.

In [ ]:
def upload_shards(
    bucket: storage.Bucket,
    source_root: str,
    pattern: str,
    destination_prefix: str = "embeddings",
    batch_size: int = 20,
) -> None:
    shard_paths = list_shards(f"{source_root}/{pattern}")
    if not shard_paths:
        raise FileNotFoundError(f"No shards found matching {pattern}")

    for idx in range(0, len(shard_paths), batch_size):
        batch = shard_paths[idx : idx + batch_size]
        for shard_path in tqdm(batch, desc=f"Uploading batch {idx//batch_size + 1}"):
            blob_name = f"{destination_prefix}/{Path(shard_path).name}"
            blob = bucket.blob(blob_name)
            blob.upload_from_filename(shard_path)
        print(f"Completed batch {idx//batch_size + 1} ({len(batch)} files)")


upload_shards(bucket, LOCAL_DATA_ROOT, SHARD_PATTERN)

## 6. Verify Stored Objects
List uploaded artifacts, compute aggregate stats, and double-check that each shard is present.

In [ ]:
from math import ceil


def summarize_bucket(bucket: storage.Bucket, prefix: str = "embeddings") -> None:
    blobs = list(bucket.list_blobs(prefix=prefix))
    total_objects = len(blobs)
    total_bytes = sum(blob.size for blob in blobs)
    print(f"Objects under '{prefix}': {total_objects}")
    print(f"Total size: {total_bytes / 1e6:.2f} MB")
    if blobs:
        preview = [blob.name for blob in blobs[: min(5, total_objects)]]
        print("Sample objects:")
        for name in preview:
            print(" -", name)


summarize_bucket(bucket)

## 7. End-to-End Training & Inference Pipeline
The following cells orchestrate dataset loading, embedding generation, feature engineering, model training, evaluation, and submission file creation.

from pathlib import Path

DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("outputs")
FEATURE_DIR = Path("features")
MODEL_DIR = Path("models")
SUBMISSION_PATH = MODEL_DIR / "test_out.csv"

TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"  # Apache 2.0
IMAGE_MODEL_NAME = "laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K"  # MIT licensed
MAX_TEXT_BATCH = 2048
MAX_IMAGE_BATCH = 64

LIGHTGBM_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.03,
    "num_leaves": 512,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "min_data_in_leaf": 20,
    "lambda_l1": 0.05,
    "lambda_l2": 0.2,
    "max_bin": 255,
    "device_type": "gpu",
    "gpu_platform_id": 0,
    "gpu_device_id": 0,
    "verbose": -1,
}

for directory in (OUTPUT_DIR, FEATURE_DIR, MODEL_DIR):
    directory.mkdir(exist_ok=True)

DATA_DIR.exists(), OUTPUT_DIR.exists()

In [1]:
# Cell intentionally kept for reference. All active configuration now lives in Cell 24.
DATA_DIR.exists(), OUTPUT_DIR.exists()

NameError: name 'DATA_DIR' is not defined

### 7.2 Load Raw Datasets

In [ ]:
import pandas as pd

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

### 7.3 Generate Text Embeddings
Uses `SentenceTransformer` with batching to create `.npy` embeddings for train and test splits.

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

TEXT_TRAIN_PATH = OUTPUT_DIR / "train_text_embeddings.npy"
TEXT_TEST_PATH = OUTPUT_DIR / "test_text_embeddings.npy"

if not TEXT_TRAIN_PATH.exists() or not TEXT_TEST_PATH.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    text_model = SentenceTransformer(TEXT_MODEL_NAME, device=device)

    def encode_texts(texts: pd.Series, output_path: Path, batch_size: int) -> None:
        sentences = texts.fillna("").tolist()
        embeddings = []
        for start in range(0, len(sentences), batch_size):
            batch = sentences[start : start + batch_size]
            emb = text_model.encode(
                batch,
                batch_size=batch_size,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=False,
            )
            embeddings.append(emb)
        stacked = np.vstack(embeddings)
        np.save(output_path, stacked)
        print(f"Saved {output_path} with shape {stacked.shape}")

    encode_texts(train_df["catalog_content"], TEXT_TRAIN_PATH, MAX_TEXT_BATCH)
    encode_texts(test_df["catalog_content"], TEXT_TEST_PATH, MAX_TEXT_BATCH)
else:
    print("Text embeddings already exist; skipping generation.")

import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

IMAGE_TRAIN_PATH = OUTPUT_DIR / "train_image_embeddings.npy"
IMAGE_TEST_PATH = OUTPUT_DIR / "test_image_embeddings.npy"
IMAGES_DIR = Path("data/images")
IMAGES_TEST_DIR = Path("data/images_test")

if not IMAGE_TRAIN_PATH.exists() or not IMAGE_TEST_PATH.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model = CLIPModel.from_pretrained(IMAGE_MODEL_NAME).to(device)
    clip_processor = CLIPProcessor.from_pretrained(IMAGE_MODEL_NAME)

    def encode_images(df: pd.DataFrame, root: Path, output_path: Path) -> None:
        embeddings = []
        paths = []
        for link in df["image_link"].fillna(""):
            name = Path(link.split("?", 1)[0]).name
            paths.append(root / name)
        for start in range(0, len(paths), MAX_IMAGE_BATCH):
            batch_paths = paths[start : start + MAX_IMAGE_BATCH]
            images = []
            valid_mask = []
            for path in batch_paths:
                if path.exists():
                    try:
                        images.append(Image.open(path).convert("RGB"))
                        valid_mask.append(True)
                    except Exception:
                        images.append(Image.new("RGB", (224, 224)))
                        valid_mask.append(False)
                else:
                    images.append(Image.new("RGB", (224, 224)))
                    valid_mask.append(False)
            inputs = clip_processor(images=images, return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                outputs = clip_model.get_image_features(**inputs)
            out = outputs.cpu().numpy()
            out[~np.array(valid_mask)] = 0.0
            embeddings.append(out)
        stacked = np.vstack(embeddings)
        np.save(output_path, stacked)
        print(f"Saved {output_path} with shape {stacked.shape}")

    encode_images(train_df, IMAGES_DIR, IMAGE_TRAIN_PATH)
    encode_images(test_df, IMAGES_TEST_DIR, IMAGE_TEST_PATH)
else:
    print("Image embeddings already exist; skipping generation.")

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

IMAGE_TRAIN_PATH = OUTPUT_DIR / "train_image_embeddings.npy"
IMAGE_TEST_PATH = OUTPUT_DIR / "test_image_embeddings.npy"
IMAGES_DIR = Path("data/images")
IMAGES_TEST_DIR = Path("data/images_test")

if not IMAGE_TRAIN_PATH.exists() or not IMAGE_TEST_PATH.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if device == "cuda" else torch.float32
    clip_model = CLIPModel.from_pretrained(IMAGE_MODEL_NAME, torch_dtype=torch_dtype).to(device)
    clip_model.eval()
    clip_processor = CLIPProcessor.from_pretrained(IMAGE_MODEL_NAME)

    def encode_images(df: pd.DataFrame, root: Path, output_path: Path) -> None:
        features = []
        paths = []
        for link in df["image_link"].fillna(""):
            name = Path(link.split("?", 1)[0]).name
            paths.append(root / name)

        for start in range(0, len(paths), MAX_IMAGE_BATCH):
            batch_paths = paths[start : start + MAX_IMAGE_BATCH]
            images = []
            valid_mask = []
            for path in batch_paths:
                if path.exists():
                    try:
                        images.append(Image.open(path).convert("RGB"))
                        valid_mask.append(True)
                    except Exception:
                        images.append(Image.new("RGB", (224, 224)))
                        valid_mask.append(False)
                else:
                    images.append(Image.new("RGB", (224, 224)))
                    valid_mask.append(False)

            inputs = clip_processor(images=images, return_tensors="pt", padding=True)
            pixel_values = inputs["pixel_values"].to(device=device, dtype=torch_dtype, non_blocking=True)
            with torch.no_grad():
                outputs = clip_model.get_image_features(pixel_values=pixel_values)

            batch_features = outputs.cpu().float().numpy()
            mask = np.array(valid_mask, dtype=bool)
            batch_features[~mask] = 0.0
            features.append(batch_features)

            del inputs, pixel_values, outputs
            if device == "cuda":
                torch.cuda.empty_cache()

        stacked = np.vstack(features)
        np.save(output_path, stacked)
        print(f"Saved {output_path} with shape {stacked.shape}")

    encode_images(train_df, IMAGES_DIR, IMAGE_TRAIN_PATH)
    encode_images(test_df, IMAGES_TEST_DIR, IMAGE_TEST_PATH)
else:
    print("Image embeddings already exist; skipping generation.")

### 7.5 Feature Engineering & Matrix Assembly

In [ ]:
import re
from sklearn.preprocessing import StandardScaler

train_text = np.load(TEXT_TRAIN_PATH)
test_text = np.load(TEXT_TEST_PATH)
train_img = np.load(IMAGE_TRAIN_PATH) if IMAGE_TRAIN_PATH.exists() else np.zeros((train_text.shape[0], 0))
test_img = np.load(IMAGE_TEST_PATH) if IMAGE_TEST_PATH.exists() else np.zeros((test_text.shape[0], 0))

IPQ_REGEX = re.compile(r"(\d+\.?\d*)\s*(pack|ct|count|pcs|pc|piece|pieces)", re.IGNORECASE)


def extract_ipq(series: pd.Series) -> np.ndarray:
    values = []
    for text in series.fillna(""):
        match = IPQ_REGEX.search(text)
        if match:
            values.append(float(match.group(1)))
        else:
            values.append(0.0)
    return np.array(values, dtype=np.float32)

train_ipq = extract_ipq(train_df["catalog_content"])
test_ipq = extract_ipq(test_df["catalog_content"])

scaler = StandardScaler()
train_ipq_scaled = scaler.fit_transform(train_ipq.reshape(-1, 1))
test_ipq_scaled = scaler.transform(test_ipq.reshape(-1, 1))

train_stats = np.stack([
    train_df["catalog_content"].fillna("").str.len().to_numpy(),
    train_df["catalog_content"].fillna("").str.count(r"\d").to_numpy(),
], axis=1)

test_stats = np.stack([
    test_df["catalog_content"].fillna("").str.len().to_numpy(),
    test_df["catalog_content"].fillna("").str.count(r"\d").to_numpy(),
], axis=1)

stats_scaler = StandardScaler()
train_stats_scaled = stats_scaler.fit_transform(train_stats)
test_stats_scaled = stats_scaler.transform(test_stats)

train_features = np.concatenate([train_text, train_img, train_stats_scaled, train_ipq_scaled], axis=1)
test_features = np.concatenate([test_text, test_img, test_stats_scaled, test_ipq_scaled], axis=1)

np.save(FEATURE_DIR / "train_features.npy", train_features)
np.save(FEATURE_DIR / "test_features.npy", test_features)
np.save(FEATURE_DIR / "train_price.npy", train_df["price"].to_numpy(dtype=np.float32))

train_features.shape, test_features.shape

### 7.6 Cross-Validation & Model Training

In [ ]:
import joblib
import lightgbm as lgb
from sklearn.model_selection import KFold

features = np.load(FEATURE_DIR / "train_features.npy")
target = np.load(FEATURE_DIR / "train_price.npy")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros_like(target, dtype=np.float32)
models = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(features, target), start=1):
    train_data = lgb.Dataset(features[train_idx], label=target[train_idx])
    valid_data = lgb.Dataset(features[valid_idx], label=target[valid_idx])

    model = lgb.train(
        LIGHTGBM_PARAMS,
        train_data,
        num_boost_round=5000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=True),
            lgb.log_evaluation(period=100),
        ],
    )
    preds = model.predict(features[valid_idx], num_iteration=model.best_iteration)
    oof_preds[valid_idx] = preds
    models.append(model)
    joblib.dump(model, MODEL_DIR / f"lightgbm_fold{fold}.joblib")

np.save(MODEL_DIR / "oof_predictions.npy", oof_preds)
models

### 7.7 Evaluate SMAPE

In [ ]:
def smape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-9) -> float:
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom < eps, eps, denom)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)

score = smape(target, oof_preds)
print(f"OOF SMAPE: {score:.2f}%")

### 7.8 Fit Final Model & Export Submission

In [ ]:
final_train_data = lgb.Dataset(features, label=target)
mean_best_iteration = int(np.mean([model.best_iteration for model in models]))
mean_best_iteration = max(mean_best_iteration, 100)
print(f"Using {mean_best_iteration} boosting rounds for final model")
final_model = lgb.train(
    LIGHTGBM_PARAMS,
    final_train_data,
    num_boost_round=mean_best_iteration,
)
joblib.dump(final_model, MODEL_DIR / "lightgbm_final.joblib")

submission_preds = final_model.predict(test_features)
submission_preds = np.clip(submission_preds, a_min=0.1, a_max=None)
submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": submission_preds,
})
submission.to_csv(SUBMISSION_PATH, index=False)
print("Submission saved to", SUBMISSION_PATH)